In [ ]:
#| default_exp build

In [ ]:
#| export
import numpy as np
from litesearch.core import content_id, upsert_all, sql_in
from litesearch import DTYPE
from vruksha.entities import norm_ent, extract_typed, _lex_ok

def graph_write(db, chunks, results, store='store', prefix=None, seed_fn=None, canon=None):
    "Upsert entities/mentions/edges from typed results. `seed_fn`/`canon` add and canonicalize a corpus's structured citations. Upsert, so it augments."
    from fastcore.all import first
    g = db.get_graph(store, prefix); by = {c['id']:c for c in chunks}
    ents, mens, edges = {}, {}, {}
    def ent(name, kind='concept'):
        nm = norm_ent(name, canon); eid = content_id(nm)
        ents.setdefault(eid, dict(content=nm, kind=kind, canon=eid, freq=0)); ents[eid]['freq'] += 1; return eid
    def mention(cid, eid, s): mens[(cid,eid)] = dict(chunk_id=cid, entity_id=eid, surface=(s or '')[:80], n=1)
    def edge(s, d, rel):
        if s and d and s != d: edges[(s,d,rel)] = dict(src=s, dst=d, rel=rel, weight=1.0, n=1)
    for r in results:
        cid = r['id']; c = by.get(cid) or {}
        for e in r.get('entities',[]): mention(cid, ent(e['name'], e.get('type') or 'concept'), e['name'])
        for rel in r.get('relations',[]): edge(ent(rel['src']), ent(rel['dst']), rel['rel'])
        if seed_fn:
            own = set(seed_fn(c.get('heading') or '')); anchor = ent(first(own), 'citation') if own else None
            if anchor: mention(cid, anchor, first(own))
            for a in seed_fn(c.get('content') or ''):
                if a in own: continue
                d = ent(a, 'citation'); mention(cid, d, a); edge(anchor or cid, d, 'refers_to')
    if ents: g.entities.insert_all(list(ents.values()), upsert=True, hash_id='id', hash_id_columns=['content'])
    if mens: upsert_all(g.mentions, list(mens.values()), ('chunk_id','entity_id'))
    if edges: upsert_all(g.edges, list(edges.values()), ('src','dst','rel'))
    return dict(entities=len(ents), mentions=len(mens), edges=len(edges))

def embed_entities(db, store='store', emb_fn=None, prefix=None):
    "Embed entity names that lack a vector, then rebuild the entity ANN index. Incremental."
    assert emb_fn, 'emb_fn is required'
    g = db.get_graph(store, prefix); rows = [r for r in g.entities(select='id, content, embedding') if not r['embedding']]
    for r, v in zip(rows, emb_fn([r['content'] for r in rows]) if rows else []):
        g.entities.update(dict(id=r['id'], embedding=np.asarray(v, dtype=DTYPE).tobytes()))
    g.entities.rebuild_index(); return len(rows)

def prime(db, store, emb, k=12, prefix=None):
    "Existing entity names nearest a chunk vector, so the model reuses them instead of duplicating."
    g = db.get_graph(store, prefix); m = db._ann_meta(f'{g.prefix}entities')
    if not (m and m['ndim']): return []
    return [r['content'] for r in g.entities.ann_search(emb, columns=['content'], limit=k, dtype=DTYPE)]

def new_chunk_ids(db, store='store', prefix=None):
    "Chunk ids with no non-topic mention: what an incremental `build_graph` still has to extract."
    g = db.get_graph(store, prefix); p = g.prefix
    seen = {r['c'] for r in db.q(f"select distinct m.chunk_id c from {p}mentions m "
                                 f"join {p}entities e on e.id=m.entity_id where e.kind!='topic'")}
    return [r['id'] for r in db.t[store](select='id') if r['id'] not in seen]

def _uf(parent, x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
def resolve_entities(db, store='store', prefix=None, k=8):
    "Merge near-duplicate entities priming still split: ANN proposes, `_lex_ok` decides, mentions/edges repoint. Conservative by design."
    g = db.get_graph(store, prefix); p = g.prefix
    ents = {r['id']:r for r in g.entities(select='id, rowid as rowid, content, kind, freq, embedding') if r['embedding']}
    if len(ents) < 2: return dict(merged=0)
    parent = {i:i for i in ents}
    for i, r in ents.items():
        for n in g.entities.ann_neighbors(r['rowid'], limit=k, columns=['id'], dtype=DTYPE):
            j = n['id']
            if j in ents and j != i and ents[i]['kind'] == ents[j]['kind'] and _lex_ok(ents[i]['content'], ents[j]['content']):
                parent[_uf(parent,i)] = _uf(parent,j)
    groups = {}
    for i in ents: groups.setdefault(_uf(parent,i), []).append(i)
    merged = 0
    for grp in groups.values():
        if len(grp) < 2: continue
        canon = max(grp, key=lambda x: (ents[x]['freq'], -len(ents[x]['content'])))
        for x in grp:
            if x == canon: continue
            db.execute(f"delete from {p}mentions where entity_id={x!r} and chunk_id in (select chunk_id from {p}mentions where entity_id={canon!r})")
            db.execute(f"update {p}mentions set entity_id={canon!r} where entity_id={x!r}")
            db.execute(f"update or ignore {p}edges set src={canon!r} where src={x!r}")
            db.execute(f"update or ignore {p}edges set dst={canon!r} where dst={x!r}")
            db.execute(f"delete from {p}edges where src={x!r} or dst={x!r}")
            g.entities.delete_where(f"id={x!r}"); merged += 1
    db.execute(f"delete from {p}edges where src=dst")
    return dict(merged=merged)

def build_graph(db, chat, emb_fn, store='store', prefix=None, batch=200, seed_fn=None, canon=None, prime_k=12, resolve=True):
    "Typed graph over new chunks of a litesearch store: prime, extract via `chat`, write, embed, resolve. Incremental; caller injects chat and emb_fn."
    g = db.get_graph(store, prefix); new = new_chunk_ids(db, store, prefix)
    if not new: return dict(entities=0, mentions=0, edges=0, new=0)
    cols = 'id, heading, content' if 'heading' in {c.name for c in db.t[store].columns} else 'id, content'
    tot = dict(entities=0, mentions=0, edges=0, new=len(new))
    for i in range(0, len(new), batch):
        chunks = [dict(id=r['id'], heading=r.get('heading'), content=r['content'])
                  for r in db.t[store](select=cols, where=sql_in('id', new[i:i+batch]))]
        vecs = emb_fn([c['content'] for c in chunks])
        hints = [', '.join(prime(db, store, np.asarray(v, dtype=DTYPE).tobytes(), prime_k, prefix)) or None for v in vecs]
        w = graph_write(db, chunks, extract_typed(chunks, chat, hints), store, prefix, seed_fn, canon)
        for key in ('entities','mentions','edges'): tot[key] += w[key]
        embed_entities(db, store, emb_fn, prefix)
    if resolve: tot['merged'] = resolve_entities(db, store, prefix).get('merged', 0)
    return tot

In [ ]:
#| hide
import numpy as np, hashlib
from litesearch import database
def _femb(ts): return [np.frombuffer(hashlib.sha256(t.encode()).digest()[:16], dtype=np.uint8).astype(np.float16)/255 for t in ts]
class _Chat:
    def oneshot(self, u, **k):
        return ('{"summary":"s","entities":[{"name":"transformer","type":"model"},{"name":"attention","type":"concept"}],'
                '"relations":[{"src":"transformer","rel":"uses","dst":"attention"}]}')
db = database(); st = db.get_store(name='store', hash=True, ann=True, node_id=str)
rows = [dict(content=f'chunk {i} on transformer and attention', node_id=f'd#{i}') for i in range(3)]
for r,v in zip(rows, _femb([r['content'] for r in rows])): r['embedding'] = v.tobytes()
st.insert_all(rows, upsert=True, hash_id='id', hash_id_columns=['content']); st.rebuild_index()
assert len(new_chunk_ids(db,'store')) == 3
res = build_graph(db, _Chat(), _femb, store='store', batch=8)
assert res['entities'] >= 2 and res['edges'] >= 1
assert new_chunk_ids(db,'store') == []                        # incremental: nothing left to extract
g = db.get_graph('store'); eids = {r['id'] for r in g.entities(select='id')}
assert all(m['entity_id'] in eids for m in g.mentions(select='entity_id'))   # no dangling after resolve